# Appendix 01 — Constant Memory & Events

> Independent appendix, distilled from *CUDA by Example* (Sanders & Kandrot),
> **Chapter 6 — Constant Memory and Events**. Not tied to one course chapter; it
> rounds out the memory-space toolkit and formalizes the timing method you've been
> using since Chapter 9.

Two small, useful tools:

1. **Constant memory** (`__constant__`) — a 64 KB read-only region that is *cached* and *broadcast*: when every thread in a warp reads the **same** address, the hardware serves all 32 from a single cache read. Perfect for small, read-only parameter tables that every thread consults.
2. **CUDA events** (`cudaEvent_t`) — the right way to time GPU work. You've used `cudaEventRecord` / `cudaEventElapsedTime` in every benchmark; here we explain *why* it's correct and CPU timers aren't.

The book demonstrates both with a tiny **ray tracer**: a scene of spheres, read by every pixel-thread. We keep the computation and drop the on-screen rendering (no OpenGL).

### Learning objectives

By the end you will:

- Declare and fill `__constant__` memory with `cudaMemcpyToSymbol`, and explain the **warp-broadcast** win.
- Say when constant memory helps (uniform reads) and when it *hurts* (divergent reads).
- Use `cudaEvent_t` to time a kernel correctly, and explain why you must `cudaEventSynchronize` before reading the elapsed time.


## 1. Concept — Constant Memory

`__constant__` declares a variable in a special 64 KB region of device memory:

```c
__constant__ Sphere s[SPHERES];                 // file-scope, device-side
...
cudaMemcpyToSymbol(s, h_spheres, sizeof(h_spheres));   // host fills it (note: ToSymbol, not a pointer)
```

Two properties make it fast for the right access pattern:

- **Cached**: reads come from a constant cache, not DRAM, after the first touch.
- **Broadcast**: when all 32 threads of a warp read the *same* constant address, the hardware satisfies them with **one** read and broadcasts the value. That's a 32× reduction in memory traffic versus 32 separate global loads.

The catch — and it's important: if threads in a warp read *different* constant addresses, those reads **serialize**. Constant memory is a win only for *uniform* reads (every thread wants the same value at the same time), which is exactly the case for a small parameter table consulted identically by all threads.


## 2. Demo — Ray Tracing a Scene of Spheres

Classic book example: a grid of pixel-threads each shoots a ray straight into the scene and finds the nearest sphere it hits, producing that pixel's brightness. Every thread loops over **all** spheres — and at each step of that loop, all threads in a warp read the *same* sphere's fields. That's the uniform-read pattern constant memory loves.

We build it twice — spheres in `__constant__` vs. spheres in ordinary global memory — time both with CUDA events, and confirm the images are identical.


In [ ]:
!mkdir -p course/appendix01_build


In [ ]:
%%writefile course/appendix01_build/raytrace.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define DIM     1024
#define SPHERES 200
#define rnd(x)  ((x) * (float)rand() / RAND_MAX)
#define INF     2e10f

struct Sphere {
    float r, b, g;
    float radius;
    float x, y, z;
    // returns z of the hit (for depth test) and writes shading factor n; -INF if miss
    __device__ float hit(float ox, float oy, float* n) const {
        float dx = ox - x, dy = oy - y;
        if (dx*dx + dy*dy < radius*radius) {
            float dz = sqrtf(radius*radius - dx*dx - dy*dy);
            *n = dz / radius;
            return dz + z;
        }
        return -INF;
    }
};

__constant__ Sphere s_const[SPHERES];      // constant-memory copy of the scene

__device__ void render(const Sphere* s, float* out) {
    int x = threadIdx.x + blockIdx.x * blockDim.x;
    int y = threadIdx.y + blockIdx.y * blockDim.y;
    if (x >= DIM || y >= DIM) return;
    int offset = x + y * DIM;
    float ox = (x - DIM/2), oy = (y - DIM/2);
    float r = 0, g = 0, b = 0, maxz = -INF;
    for (int i = 0; i < SPHERES; i++) {        // uniform read of s[i] across the warp
        float n, t = s[i].hit(ox, oy, &n);
        if (t > maxz) { float fscale = n; r = s[i].r*fscale; g = s[i].g*fscale; b = s[i].b*fscale; maxz = t; }
    }
    out[offset] = r + g + b;                    // brightness; we checksum this
}

__global__ void ray_const(float* out)              { render(s_const, out); }
__global__ void ray_global(const Sphere* s, float* out) { render(s, out); }

int main(void) {
    Sphere* h = (Sphere*)malloc(sizeof(Sphere) * SPHERES);
    srand(1234);
    for (int i = 0; i < SPHERES; i++) {
        h[i].r = rnd(1.0f); h[i].g = rnd(1.0f); h[i].b = rnd(1.0f);
        h[i].x = rnd(1000.0f) - 500; h[i].y = rnd(1000.0f) - 500; h[i].z = rnd(1000.0f) - 500;
        h[i].radius = rnd(100.0f) + 20;
    }

    float *d_out; cudaMalloc(&d_out, DIM*DIM*sizeof(float));
    Sphere* d_s;  cudaMalloc(&d_s, sizeof(Sphere)*SPHERES);
    cudaMemcpy(d_s, h, sizeof(Sphere)*SPHERES, cudaMemcpyHostToDevice);
    cudaMemcpyToSymbol(s_const, h, sizeof(Sphere)*SPHERES);   // fill constant memory

    dim3 block(16, 16), grid((DIM+15)/16, (DIM+15)/16);
    cudaEvent_t s0, s1; cudaEventCreate(&s0); cudaEventCreate(&s1);
    float* img = (float*)malloc(DIM*DIM*sizeof(float));
    int iters = 100;

    // --- constant memory ---
    ray_const<<<grid, block>>>(d_out); cudaDeviceSynchronize();
    cudaEventRecord(s0);
    for (int k = 0; k < iters; k++) ray_const<<<grid, block>>>(d_out);
    cudaEventRecord(s1); cudaEventSynchronize(s1);
    float ms_c; cudaEventElapsedTime(&ms_c, s0, s1);
    cudaMemcpy(img, d_out, DIM*DIM*sizeof(float), cudaMemcpyDeviceToHost);
    double sum_c = 0; for (int i = 0; i < DIM*DIM; i++) sum_c += img[i];

    // --- global memory ---
    ray_global<<<grid, block>>>(d_s, d_out); cudaDeviceSynchronize();
    cudaEventRecord(s0);
    for (int k = 0; k < iters; k++) ray_global<<<grid, block>>>(d_s, d_out);
    cudaEventRecord(s1); cudaEventSynchronize(s1);
    float ms_g; cudaEventElapsedTime(&ms_g, s0, s1);
    cudaMemcpy(img, d_out, DIM*DIM*sizeof(float), cudaMemcpyDeviceToHost);
    double sum_g = 0; for (int i = 0; i < DIM*DIM; i++) sum_g += img[i];

    printf("image checksum  constant=%.3f  global=%.3f  -> %s\n",
           sum_c, sum_g, (fabs(sum_c - sum_g) < 1e-3) ? "IDENTICAL" : "DIFFER");
    printf("constant memory : %6.3f ms/frame\n", ms_c/iters);
    printf("global memory   : %6.3f ms/frame\n", ms_g/iters);
    printf("constant is %.2fx the speed of global\n", ms_g/ms_c);

    free(h); free(img); cudaFree(d_out); cudaFree(d_s);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix01_build/raytrace course/appendix01_build/raytrace.cu && ./course/appendix01_build/raytrace


The two images are **identical** (same math, different memory space). The constant-memory version is usually a little faster because every warp's read of `s[i]` is a single broadcast instead of 32 global loads — but on modern GPUs the L1/L2 caches also absorb the global version's uniform reads, so the gap is smaller than on the 2010-era hardware the book used. The point stands: for small, uniformly-read, read-only tables, constant memory is the purpose-built tool.

> A note for honesty: on Ada (the 4080), the read-only global path is cached so well that constant memory's edge can be within noise or even reversed depending on occupancy. The *mechanism* (broadcast for uniform reads) is what to remember; don't treat the exact ratio as a law.


## 3. Concept — CUDA Events for Timing

Kernel launches are **asynchronous**: `kernel<<<...>>>()` returns to the CPU immediately, before the GPU finishes. So a CPU stopwatch around the launch measures *launch overhead*, not *execution time*. The fix is to timestamp on the **GPU's** timeline with events:

```c
cudaEvent_t start, stop;
cudaEventCreate(&start); cudaEventCreate(&stop);
cudaEventRecord(start);                 // enqueue a timestamp
kernel<<<grid, block>>>(...);           // enqueue the work
cudaEventRecord(stop);                  // enqueue another timestamp
cudaEventSynchronize(stop);             // WAIT until 'stop' has actually happened
float ms; cudaEventElapsedTime(&ms, start, stop);
```

Three rules this encodes:

1. **Record on the stream**, so the timestamps sit in the same queue as the kernel and bracket its real execution.
2. **`cudaEventSynchronize(stop)`** before reading — the events are enqueued asynchronously too, so you must block until `stop` is reached or you'll read garbage.
3. **Warm up + average** over many iterations (we do `iters` loops): the first launch pays one-time costs (context, JIT), so time the steady state.

This is the exact recipe behind every benchmark cell in the course — Chapter 9's bandwidth sweep, Chapter 11's `float4` comparison, Chapter 13b's histogram speedup.


## 4. Translation Bridge

| Book (Ch6) | Where it shows up | Note |
|---|---|---|
| `__constant__ Sphere s[]` | small read-only params broadcast to all threads | `llm.c` mostly relies on `__restrict__` + the read-only data cache (`__ldg` / `const __restrict__`) instead of `__constant__`, because its read-only tensors exceed 64 KB |
| `cudaMemcpyToSymbol` | one-time upload of a constant table | symbol copy, not a device pointer — a common gotcha |
| `cudaEvent_t` timing | **every** `llm.c` benchmark + `dev/cuda/*` test harnesses | the canonical GPU timing method |
| warp **broadcast** | uniform reads of shared parameters | the same idea that makes a warp reading one address cheap |

The honest `llm.c` connection: it rarely uses `__constant__` (its weight tensors are far larger than 64 KB), but it leans hard on the *idea* behind it — keep read-only data in the fastest cache the access pattern allows — and on `cudaEvent` timing everywhere.


## 5. Common Pitfalls

- **`cudaMemcpyToSymbol`, not `cudaMemcpy`** — constant memory is a *symbol*, not a pointer you `cudaMalloc`. Passing `&s_const` to `cudaMemcpy` is a classic bug.
- **64 KB hard limit** — `__constant__` can't hold a weight matrix; it's for small tables.
- **Divergent constant reads serialize** — if warp threads read different constant addresses, you lose the broadcast and it can be *slower* than global. Constant memory ≠ "free fast memory."
- **Forgetting `cudaEventSynchronize`** before `cudaEventElapsedTime` → meaningless timings.
- **Timing the first launch** — always warm up; the first call includes one-time setup.


## 6. TODO Exercise — Time a Kernel Correctly

Below, a SAXPY kernel (`y = a*x + y`) is timed — but the timing code is incomplete. Fill in the two TODOs so it reports correct steady-state milliseconds.


In [ ]:
%%writefile course/appendix01_build/exercise1.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void saxpy(float a, const float* x, float* y, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) y[i] = a * x[i] + y[i];
}

int main(void) {
    const int N = 1 << 24;
    float *x, *y; cudaMalloc(&x, N*4); cudaMalloc(&y, N*4);
    cudaMemset(x, 0, N*4); cudaMemset(y, 0, N*4);
    int block = 256, grid = (N + block - 1) / block, iters = 100;

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    saxpy<<<grid, block>>>(2.0f, x, y, N); cudaDeviceSynchronize();   // warmup

    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) saxpy<<<grid, block>>>(2.0f, x, y, N);
    // TODO 1: record the stop event
    // TODO 2: synchronize on the stop event before reading the time
    float ms = 0.0f;
    cudaEventElapsedTime(&ms, s, e);
    printf("saxpy: %.4f ms/iter  -> %s\n", ms/iters, (ms > 0.0f) ? "OK" : "BROKEN (0 ms)");
    cudaFree(x); cudaFree(y);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix01_build/exercise1 course/appendix01_build/exercise1.cu && ./course/appendix01_build/exercise1


### Solution

In [ ]:
%%writefile course/appendix01_build/exercise1_sol.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void saxpy(float a, const float* x, float* y, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    if (i < n) y[i] = a * x[i] + y[i];
}

int main(void) {
    const int N = 1 << 24;
    float *x, *y; cudaMalloc(&x, N*4); cudaMalloc(&y, N*4);
    cudaMemset(x, 0, N*4); cudaMemset(y, 0, N*4);
    int block = 256, grid = (N + block - 1) / block, iters = 100;

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    saxpy<<<grid, block>>>(2.0f, x, y, N); cudaDeviceSynchronize();   // warmup

    cudaEventRecord(s);
    for (int k = 0; k < iters; k++) saxpy<<<grid, block>>>(2.0f, x, y, N);
    cudaEventRecord(e);              // TODO 1
    cudaEventSynchronize(e);         // TODO 2
    float ms = 0.0f;
    cudaEventElapsedTime(&ms, s, e);
    printf("saxpy: %.4f ms/iter  -> %s\n", ms/iters, (ms > 0.0f) ? "OK" : "BROKEN (0 ms)");
    cudaFree(x); cudaFree(y);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/appendix01_build/exercise1_sol course/appendix01_build/exercise1_sol.cu && ./course/appendix01_build/exercise1_sol


## Recap

- **Constant memory** (`__constant__`, 64 KB, filled via `cudaMemcpyToSymbol`) is cached and **broadcasts** a single read to all 32 warp threads — fast for *uniform* reads, slow for divergent ones.
- Modern GPUs cache uniform global reads well too, so constant memory's edge is smaller than in the book's era — but the broadcast idea is fundamental.
- **CUDA events** are the correct GPU timer: record around the work *on the stream*, `cudaEventSynchronize` before reading, warm up and average. This is the method behind every benchmark in the course.

### What's next

**Appendix 02 — Streams & Async Overlap** (book Ch10): pinned memory, `cudaMemcpyAsync`, and overlapping copy with compute so the GPU isn't idle waiting on PCIe — the ideas that inform the data-loading pipeline in Chapter 19.
